Exercise 9 — Predicting Number of College Applications
In this exercise, we will predict the number of applications received using the other variables in the **College** data set.


(a) Split the data set into a training set and a test set.

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1) Load data
college = pd.read_csv('/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Textbook Dataset/College.csv')

# 2) Define X (predictors) and y (target)
y = college["Apps"]
X = college.drop(columns=["Apps", "Unnamed: 0"])  # drop target + school name column

# 3) Identify categorical + numerical columns
cat_cols = ["Private"]                      # Yes/No
num_cols = [c for c in X.columns if c not in cat_cols]

# 4) Preprocess: one-hot encode Private, standardize numeric vars
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

# 5) Train/test split (50/50 like ISLR labs)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=1
)

print(X_train.shape, X_test.shape)


(388, 17) (389, 17)


(b) Fit a linear model using least squares on the training set, and report the test error obtained.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ols_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

ols_model.fit(X_train, y_train)
ols_pred = ols_model.predict(X_test)

ols_mse = mean_squared_error(y_test, ols_pred)
print("OLS Test MSE:", ols_mse)
print("OLS Test RMSE:", np.sqrt(ols_mse))


OLS Test MSE: 1425055.5873112206
OLS Test RMSE: 1193.7569213668337


(c) Fit a **ridge regression** model on the training set, with λ chosen by cross-validation. Report the test error obtained.

In [17]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge())
])

alphas = np.logspace(-3, 5, 200)  # candidate lambdas

ridge_cv = GridSearchCV(
    ridge_pipe,
    param_grid={"model__alpha": alphas},
    scoring="neg_mean_squared_error",
    cv=10
)

ridge_cv.fit(X_train, y_train)
best_alpha_ridge = ridge_cv.best_params_["model__alpha"]

ridge_pred = ridge_cv.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_pred)

print("Best ridge alpha (λ):", best_alpha_ridge)
print("Ridge Test MSE:", ridge_mse)
print("Ridge Test RMSE:", np.sqrt(ridge_mse))


Best ridge alpha (λ): 3.1440354715915
Ridge Test MSE: 1504469.8188098765
Ridge Test RMSE: 1226.5683098832599


(d)Fit a **lasso** model on the training set, with λ chosen by cross-validation. Report the test error obtained, **along with the number of non-zero coefficient estimates**.

In [18]:
from sklearn.linear_model import Lasso

lasso_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Lasso(max_iter=20000))
])

alphas_lasso = np.logspace(-3, 3, 200)

lasso_cv = GridSearchCV(
    lasso_pipe,
    param_grid={"model__alpha": alphas_lasso},
    scoring="neg_mean_squared_error",
    cv=10
)

lasso_cv.fit(X_train, y_train)
best_alpha_lasso = lasso_cv.best_params_["model__alpha"]

lasso_pred = lasso_cv.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_pred)

# Count non-zero coefficients
best_lasso = lasso_cv.best_estimator_
coef = best_lasso.named_steps["model"].coef_
nonzero = np.sum(coef != 0)

print("Best lasso alpha (λ):", best_alpha_lasso)
print("Lasso Test MSE:", lasso_mse)
print("Lasso Test RMSE:", np.sqrt(lasso_mse))
print("Number of non-zero coefficients:", nonzero)


Best lasso alpha (λ): 10.234114021054527
Lasso Test MSE: 1410171.0134990087
Lasso Test RMSE: 1187.506216193839
Number of non-zero coefficients: 16


(e) Fit a **PCR (Principal Components Regression)** model on the training set, with **M** chosen by cross-validation.  
Report the test error obtained, along with the value of **M** selected by cross-validation.

In [19]:
from sklearn.decomposition import PCA

# number of features after preprocessing
preprocess.fit(X_train)
n_features = preprocess.transform(X_train).shape[1]

pcr_pipe = Pipeline([
    ("preprocess", preprocess),
    ("pca", PCA()),
    ("model", LinearRegression())
])

pcr_cv = GridSearchCV(
    pcr_pipe,
    param_grid={"pca__n_components": list(range(1, n_features + 1))},
    scoring="neg_mean_squared_error",
    cv=10
)

pcr_cv.fit(X_train, y_train)
best_M_pcr = pcr_cv.best_params_["pca__n_components"]

pcr_pred = pcr_cv.predict(X_test)
pcr_mse = mean_squared_error(y_test, pcr_pred)

print("Best number of PCR components (M):", best_M_pcr)
print("PCR Test MSE:", pcr_mse)
print("PCR Test RMSE:", np.sqrt(pcr_mse))


Best number of PCR components (M): 16
PCR Test MSE: 1451254.7527202028
PCR Test RMSE: 1204.6803529236304


(f) Fit a **PLS (Partial Least Squares)** model on the training set, with **M** chosen by cross-validation.  
Report the test error obtained, along with the value of **M** selected by cross-validation.

In [20]:
from sklearn.cross_decomposition import PLSRegression

pls_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", PLSRegression())
])

pls_cv = GridSearchCV(
    pls_pipe,
    param_grid={"model__n_components": list(range(1, n_features + 1))},
    scoring="neg_mean_squared_error",
    cv=10
)

pls_cv.fit(X_train, y_train)
best_M_pls = pls_cv.best_params_["model__n_components"]

pls_pred = pls_cv.predict(X_test).ravel()
pls_mse = mean_squared_error(y_test, pls_pred)

print("Best number of PLS components (M):", best_M_pls)
print("PLS Test MSE:", pls_mse)
print("PLS Test RMSE:", np.sqrt(pls_mse))


Best number of PLS components (M): 9
PLS Test MSE: 1432035.272741555
PLS Test RMSE: 1196.6767620128483


(g) Comment on the results obtained.
- How accurately can we predict the number of college applications received?  
- Is there much difference among the test errors resulting from these five approaches?

In [22]:
results = pd.DataFrame({
    "Model": ["OLS", "Ridge (CV)", "Lasso (CV)", "PCR (CV)", "PLS (CV)"],
    "Test MSE": [ols_mse, ridge_mse, lasso_mse, pcr_mse, pls_mse],
    "Test RMSE": [np.sqrt(ols_mse), np.sqrt(ridge_mse), np.sqrt(lasso_mse),
                  np.sqrt(pcr_mse), np.sqrt(pls_mse)],
    "Tuning / Notes": [
        "-",
        f"alpha={best_alpha_ridge:.3g}",
        f"alpha={best_alpha_lasso:.3g}, nonzero={nonzero}",
        f"M={best_M_pcr}",
        f"M={best_M_pls}"
    ]
})

results.sort_values("Test MSE")

,Model,Test MSE,Test RMSE,Tuning / Notes
2,Lasso (CV),1.410171e+06,1187.506216,"alpha=10.2, nonzero=16"
0,OLS,1.425056e+06,1193.756921,-
4,PLS (CV),1.432035e+06,1196.676762,M=9
3,PCR (CV),1.451255e+06,1204.680353,M=16
1,Ridge (CV),1.504470e+06,1226.568310,alpha=3.14


In [24]:
results_rounded = results.copy()
results_rounded["Test MSE"] = results_rounded["Test MSE"].round(0)
results_rounded["Test RMSE"] = results_rounded["Test RMSE"].round(1)

results_rounded

,Model,Test MSE,Test RMSE,Tuning / Notes
0,OLS,1425056.0,1193.8,-
1,Ridge (CV),1504470.0,1226.6,alpha=3.14
2,Lasso (CV),1410171.0,1187.5,"alpha=10.2, nonzero=16"
3,PCR (CV),1451255.0,1204.7,M=16
4,PLS (CV),1432035.0,1196.7,M=9


Lasso regression performs the best, achieving the lowest test MSE and RMSE while also reducing the model to 16 non-zero predictors, which improves interpretability.

OLS performs well and is only slightly worse than Lasso, while PLS also performs competitively with a moderate number of components (M = 9). PCR requires many components (M = 16) and still performs worse than Lasso and OLS. Ridge regression performs the worst, suggesting that uniform shrinkage of all coefficients may not be optimal for this problem.

Overall, Lasso provides the best balance of prediction accuracy and model simplicity.